# Music Piece Classification — MLP on MERT Embeddings

This notebook trains a one-hidden-layer MLP on frozen MERT embeddings,
extending the logistic regression baseline from the previous notebook.

Key architectural decisions:
- **Model:** `nn.Module` with one hidden layer (768 → H → 430)
- **Activation:** ReLU with Kaiming uniform initialization (correct pairing)
- **Regularization:** Dropout applied *after* the activation (standard placement)
- **Optimizer:** Adam with `ReduceLROnPlateau` schedule — more consequential than
  for the convex linear case because the MLP loss landscape is non-convex
- **No L2 normalization** — MERT embedding magnitudes carry piece-identity signal

**Pipeline assumes you have already run:**
```
python data_prep.py
python render.py
python embed.py
```

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/MyDrive/stat-4830

## 0. Imports & Hyperparameters

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import EMBEDDINGS_DIR

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Architecture ──────────────────────────────────────────────────────────────
HIDDEN_DIM   = 512      # width of the single hidden layer; try 256 or 768 later
DROPOUT_P    = 0.4      # dropout rate after ReLU; sweep 0.3–0.5 if overfitting

# ── Optimizer ─────────────────────────────────────────────────────────────────
LR           = 1e-3
WEIGHT_DECAY = 0        # L2 reg had no effect in linear model; revisit if MLP overfits

# ── Scheduler ─────────────────────────────────────────────────────────────────
# ReduceLROnPlateau monitors test top-1. If it doesn't improve for PATIENCE
# epochs the LR is multiplied by FACTOR. More consequential here than for the
# linear model because non-convex landscapes have flat regions and saddle points
# where a fixed LR can stall.
SCHEDULER_PATIENCE = 50
SCHEDULER_FACTOR   = 0.5
SCHEDULER_MIN_LR   = 1e-5

# ── Training ──────────────────────────────────────────────────────────────────
N_EPOCHS   = 1000
BATCH_SIZE = 256

# ── Evaluation ────────────────────────────────────────────────────────────────
TOP_N_CONFUSED      = 20
TOP_N_CLASSES_CHART = 40

print(f"Optimizer:  Adam | lr={LR} | weight_decay={WEIGHT_DECAY}")
print(f"Scheduler:  ReduceLROnPlateau | patience={SCHEDULER_PATIENCE} | factor={SCHEDULER_FACTOR} | min_lr={SCHEDULER_MIN_LR}")
print(f"Training:   {N_EPOCHS} epochs | batch_size={BATCH_SIZE}")
print(f"Hidden dim: {HIDDEN_DIM} | dropout: {DROPOUT_P}")

## 1. Load Embeddings

In [ ]:
# Load pre-computed MERT embeddings from disk.
X_train_np = np.load(EMBEDDINGS_DIR / "embeddings_train.npy")
X_test_np  = np.load(EMBEDDINGS_DIR / "embeddings_test.npy")

y_train_str = np.load(EMBEDDINGS_DIR / "labels_train.npy", allow_pickle=True)
y_test_str  = np.load(EMBEDDINGS_DIR / "labels_test.npy",  allow_pickle=True)

# Encode string labels → integers.
# Fit on train only. With SPLIT_STRATEGY="by_snippet" every piece appears in
# the training set by construction, so the encoder will know every test label.
le = LabelEncoder()
le.fit(y_train_str)
y_train_np = le.transform(y_train_str).astype(np.int64)
y_test_np  = le.transform(y_test_str).astype(np.int64)

n_classes = len(le.classes_)
d         = X_train_np.shape[1]  # embedding dimension (768 for MERT-95M)

print(f"Train: {X_train_np.shape[0]} snippets")
print(f"Test:  {X_test_np.shape[0]} snippets")
print(f"Embedding dim d: {d}")
print(f"Classes C:       {n_classes}")
print(f"Random baseline: {1/n_classes:.4f} ({100/n_classes:.2f}%)")

## 2. Dataset & DataLoader

Identical to the logistic regression notebook — thin numpy wrapper.

In [ ]:
class EmbeddingDataset(Dataset):
    """
    Thin wrapper around pre-computed embedding arrays.

    X: (N, d) float32 embeddings
    y: (N,)   int64 class labels
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = EmbeddingDataset(X_train_np, y_train_np)
test_dataset  = EmbeddingDataset(X_test_np,  y_test_np)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 3. Model

One-hidden-layer MLP as an `nn.Module`:

$$x \in \mathbb{R}^{768} \xrightarrow{\text{Linear}} h \in \mathbb{R}^{H} \xrightarrow{\text{ReLU}} \xrightarrow{\text{Dropout}} \xrightarrow{\text{Linear}} z \in \mathbb{R}^{430}$$

**Initialization:** Kaiming uniform for both linear layers. Kaiming accounts for the
ReLU non-linearity — it scales weights by $\sqrt{2/\text{fan\_in}}$ so that the
variance of activations is preserved through each layer at initialization. This
matters more here than for logistic regression because the gradient signal has to
flow through the hidden layer before reaching $W_1$.

**Dropout placement:** After the ReLU activation (not before). This is the standard
placement — the activation has already decided which neurons fire, and dropout then
stochastically zeros some of those active units, which is the intended regularization.

In [ ]:
class MLP(nn.Module):
    """
    One-hidden-layer MLP for multi-class classification over MERT embeddings.

    Architecture:
        Linear(d, H) → ReLU → Dropout(p) → Linear(H, C)

    Forward pass returns raw logits; the loss function handles softmax.

    NOTE: No L2 normalization — MERT embedding magnitudes carry piece-identity
    information. Normalizing discards that signal (see logistic regression notebook).
    """

    def __init__(self, d: int, hidden_dim: int, n_classes: int, dropout_p: float):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, n_classes),
        )

        # Kaiming uniform initialization — designed for ReLU activations.
        # Uses fan_in mode (default): scale = sqrt(2 / fan_in).
        # Both linear layers get Kaiming init; the bias is zeroed.
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = MLP(
    d=d,
    hidden_dim=HIDDEN_DIM,
    n_classes=n_classes,
    dropout_p=DROPOUT_P,
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters())
linear_params = d * n_classes + n_classes  # logistic regression equivalent
print(f"\nTrainable parameters: {total_params:,}")
print(f"(Linear baseline had: {linear_params:,})")
print(f"Parameter increase:   {total_params / linear_params:.1f}x")

## 4. Optimizer, Scheduler & Loss

In [ ]:
# Inverse-frequency class weighting — identical logic to the linear notebook.
counts  = np.bincount(y_train_np, minlength=n_classes).astype(np.float32)
weights = 1.0 / (counts + 1e-6)
weights = weights / weights.sum() * n_classes  # normalise so mean weight ≈ 1
class_weights = torch.from_numpy(weights).float().to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# ReduceLROnPlateau watches test top-1 accuracy ("max" mode).
# If test top-1 does not improve for PATIENCE epochs, LR is scaled by FACTOR.
# In the linear (convex) model a fixed LR was fine because Adam reliably found
# the global optimum. Here, the non-convex landscape means the optimizer can
# stall in flat regions or near saddle points — reducing LR at those moments
# helps it settle into a better basin.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=SCHEDULER_MIN_LR,
    verbose=True,
)

print(f"Optimizer: Adam | lr={LR} | weight_decay={WEIGHT_DECAY}")
print(f"Scheduler: ReduceLROnPlateau | patience={SCHEDULER_PATIENCE} | factor={SCHEDULER_FACTOR}")
print(f"Loss:      CrossEntropyLoss with inverse-frequency class weights")
print(f"Weight range: {weights.min():.3f} – {weights.max():.3f}")

## 5. Training Loop

In [ ]:
def evaluate(model, loader, device):
    """
    Run model over a DataLoader in eval mode (dropout disabled) and return:
      - mean cross-entropy loss
      - top-1 accuracy
      - top-5 accuracy
      - all predictions  (for per-class breakdown and confusion matrix)
      - all true labels
    """
    model.eval()
    total_loss   = 0.0
    correct_top1 = 0
    correct_top5 = 0
    all_preds    = []
    all_labels   = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)

            preds = logits.argmax(dim=1)
            correct_top1 += preds.eq(y_batch).sum().item()

            top5 = logits.topk(5, dim=1).indices
            correct_top5 += top5.eq(y_batch.unsqueeze(1)).any(dim=1).sum().item()

            all_preds.append(preds.cpu())
            all_labels.append(y_batch.cpu())

    n          = len(loader.dataset)
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return {
        "loss":   total_loss / n,
        "top1":   correct_top1 / n,
        "top5":   correct_top5 / n,
        "preds":  all_preds,
        "labels": all_labels,
    }

In [ ]:
history = {
    "train_loss": [], "train_top1": [], "train_top5": [],
    "test_loss":  [], "test_top1":  [], "test_top5":  [],
    "lr":         [],
}

for epoch in range(1, N_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

    # ── Evaluate ───────────────────────────────────────────────────────────
    train_metrics = evaluate(model, train_loader, DEVICE)
    test_metrics  = evaluate(model, test_loader,  DEVICE)

    # ── Step scheduler on test top-1 ───────────────────────────────────────
    # We pass test top-1 (not train loss) so the LR reduction is triggered by
    # generalisation stagnation, not just training loss plateau.
    scheduler.step(test_metrics["top1"])

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_metrics["loss"])
    history["train_top1"].append(train_metrics["top1"])
    history["train_top5"].append(train_metrics["top5"])
    history["test_loss"].append(test_metrics["loss"])
    history["test_top1"].append(test_metrics["top1"])
    history["test_top5"].append(test_metrics["top5"])
    history["lr"].append(current_lr)

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:>4}/{N_EPOCHS} | "
            f"train loss: {train_metrics['loss']:.4f}  "
            f"train top-1: {train_metrics['top1']:.3f}  "
            f"test top-1: {test_metrics['top1']:.3f}  "
            f"test top-5: {test_metrics['top5']:.3f}  "
            f"lr: {current_lr:.2e}"
        )

print(f"\nFinal test top-1: {history['test_top1'][-1]:.4f}")
print(f"Final test top-5: {history['test_top5'][-1]:.4f}")

## 6. Training Curves

In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
ax = axes[0]
ax.plot(epochs, history["train_loss"], label="Train loss", color="steelblue")
ax.plot(epochs, history["test_loss"],  label="Test loss",  color="coral", linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Loss Curve")
ax.legend()
ax.grid(alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs, history["train_top1"], label="Train top-1", color="steelblue")
ax.plot(epochs, history["test_top1"],  label="Test top-1",  color="coral",    linestyle="--")
ax.plot(epochs, history["train_top5"], label="Train top-5", color="royalblue", alpha=0.5)
ax.plot(epochs, history["test_top5"],  label="Test top-5",  color="tomato",    alpha=0.5, linestyle="--")
ax.axhline(1/n_classes, color="gray", linestyle=":", label=f"Random ({1/n_classes:.3f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Curves")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Learning rate schedule
ax = axes[2]
ax.plot(epochs, history["lr"], color="mediumpurple")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("LR Schedule (ReduceLROnPlateau)")
ax.set_yscale("log")
ax.grid(alpha=0.3)

fig.suptitle(
    f"MLP (768→{HIDDEN_DIM}→430) | Adam lr={LR} | dropout={DROPOUT_P}",
    fontsize=10
)
plt.tight_layout()
plt.savefig(EMBEDDINGS_DIR / "mlp_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Evaluation

### 7a. Final Metrics

In [ ]:
final = evaluate(model, test_loader, DEVICE)
y_pred = final["preds"].numpy()
y_true = final["labels"].numpy()

print("=" * 55)
print("FINAL TEST RESULTS — MLP")
print("=" * 55)
print(f"  Top-1 Accuracy:  {final['top1']:.4f}  ({final['top1']*100:.1f}%)")
print(f"  Top-5 Accuracy:  {final['top5']:.4f}  ({final['top5']*100:.1f}%)")
print(f"  Test Loss:       {final['loss']:.4f}")
print(f"  Random baseline: {1/n_classes:.4f}  ({100/n_classes:.2f}%)")
print(f"  Linear baseline: 0.3440  (34.4%)  [from prior notebook]")
print("=" * 55)

### 7b. Per-Class Accuracy

In [ ]:
per_class_acc = {}
for class_idx, class_name in enumerate(le.classes_):
    mask = y_true == class_idx
    if mask.sum() == 0:
        continue
    per_class_acc[class_name] = (y_pred[mask] == class_idx).mean()

sorted_classes = sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True)
accs_all = list(per_class_acc.values())

print(f"Median per-class accuracy:  {np.median(accs_all):.4f}")
print(f"Mean per-class accuracy:    {np.mean(accs_all):.4f}")
print(f"Classes with 100% accuracy: {sum(1 for v in accs_all if v == 1.0)}")
print(f"Classes with   0% accuracy: {sum(1 for v in accs_all if v == 0.0)}")
print("\nHardest pieces:")
for name, acc in sorted_classes[-10:]:
    print(f"  {name:<40} {acc:.2f}")

In [ ]:
names  = [c for c, _ in sorted_classes]
accs   = [a for _, a in sorted_classes]
n_show = min(TOP_N_CLASSES_CHART, len(sorted_classes))
half   = n_show // 2

display_names = names[:half] + names[-half:]
display_accs  = accs[:half]  + accs[-half:]
colors = ["#4CAF50"] * half + ["#F44336"] * half

fig, ax = plt.subplots(figsize=(14, max(6, n_show * 0.3)))
ax.barh(display_names, display_accs, color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(final["top1"], color="black", linestyle="--", linewidth=1.2,
           label=f"Overall top-1 ({final['top1']:.2f})")
ax.set_xlabel("Per-Class Accuracy")
ax.set_title(f"Per-Class Accuracy — Top {half} and Bottom {half} Pieces")
ax.set_xlim(0, 1.05)
ax.legend()
ax.set_yticklabels([n.split("__")[-1] for n in display_names], fontsize=8)
plt.tight_layout()
plt.savefig(EMBEDDINGS_DIR / "mlp_per_class_accuracy.png", dpi=150)
plt.show()

### 7c. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

flat_indices = np.argsort(cm_errors.ravel())[::-1][:TOP_N_CONFUSED]
top_true_idx, top_pred_idx = np.unravel_index(flat_indices, cm_errors.shape)

involved    = sorted(set(top_true_idx) | set(top_pred_idx))
sub_cm      = cm[np.ix_(involved, involved)]
names_short = [le.classes_[i].split("__")[-1] for i in involved]

fig, ax = plt.subplots(figsize=(max(10, len(involved) * 0.6),
                                max(8,  len(involved) * 0.6)))
sns.heatmap(sub_cm, xticklabels=names_short, yticklabels=names_short,
            annot=True, fmt="d", cmap="Blues", linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — {len(involved)} Most Confused Classes")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(EMBEDDINGS_DIR / "mlp_confusion_matrix.png", dpi=150)
plt.show()

print("Top confused pairs (true → predicted, count):")
for ti, pi in zip(top_true_idx, top_pred_idx):
    count = cm[ti, pi]
    if count > 0:
        print(f"  {le.classes_[ti].split('__')[-1]:<30} → "
              f"{le.classes_[pi].split('__')[-1]:<30} ({count} snippets)")

## 8. Summary

In [ ]:
print("=" * 55)
print("MLP SUMMARY")
print("=" * 55)
print(f"  Model:         MLP (d={d} → H={HIDDEN_DIM} → C={n_classes})")
print(f"  Activation:    ReLU | init: Kaiming uniform")
print(f"  Dropout:       p={DROPOUT_P} (after ReLU)")
print(f"  Optimizer:     Adam | lr={LR} | weight_decay={WEIGHT_DECAY}")
print(f"  Scheduler:     ReduceLROnPlateau | patience={SCHEDULER_PATIENCE} | factor={SCHEDULER_FACTOR}")
print(f"  Epochs:        {N_EPOCHS}")
print(f"  Parameters:    {sum(p.numel() for p in model.parameters()):,}")
print()
print(f"  Top-1 Accuracy:  {final['top1']:.4f}  ({final['top1']*100:.1f}%)")
print(f"  Top-5 Accuracy:  {final['top5']:.4f}  ({final['top5']*100:.1f}%)")
print(f"  Median per-class:{np.median(accs_all):.4f}")
print(f"  Random baseline: {1/n_classes:.4f}  ({100/n_classes:.2f}%)")
print(f"  Linear baseline: 0.3440  (34.4%)")
print("=" * 55)